In [ ]:
# Instala las librerias requeridas.

%pip install -q pandas-ta yfinance lxml

In [ ]:
# Importa las librerias requeridas.

import pandas as pd
import pandas_ta as ta

from datos import obtener_tickers

In [ ]:
# Descarga de data temporalidad 60 minutos.

tickers_60m = obtener_tickers("60m" , "3mo")

In [ ]:
# Análisis 60 minutos.

empresas_60min = []

for ticker in tickers_60m.columns.levels[0]:
    try:
        # 1.Extraemos el DataFrame individual para este ticker y limpiamos nulos
        df = tickers_60m[ticker].dropna()
        if len(df) < 50: 
            continue

        # 2. Cálculos Técnicos (aquí pandas_ta ya funcionará perfecto)
        df.ta.rsi(length=10, append=True)
        df['RSI_MA'] = ta.sma(df['RSI_10'], length=14)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # 3.Variables para validación
        rsi_act = df['RSI_10'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]
        
        # 4.Lógica de Filtrado 60min
        if (rsi_act <= 35 and rsi_act >= rsi_ant):
            empresas_60min.append({
                'Ticker': ticker,
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1),
                'MACD': round(macd_act, 3)
            })
            
    except Exception as e:
        print(f"Error en {ticker}: {e}")

In [ ]:
# Output análisis 60 minutos.

print("\n" + "="*95)
if empresas_60min:
    df_final = pd.DataFrame(empresas_60min)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales detectadas.")

In [ ]:
# Descarga de datos TEMPORALIDAD DIARIA.

tickers_1d = obtener_tickers("1d" , "1y")

In [ ]:
# ANÁLISIS EN DIARIO

empresas_diario = []

for ticker in tickers_1d.columns.levels[0]:
    try:
        df = tickers_1d[ticker].dropna()
        if len(df) < 30: continue

        # Cálculos Técnicos.
        df.ta.rsi(length=14, append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables de validación
        rsi_ant = df['RSI_14'].iloc[-2]
        rsi_act = df['RSI_14'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        

        precio_act = df['Close'].iloc[-1]

        # LÓGICA DE FILTRADO DIARIO
        estuvo_sobrevendido = rsi_act <= 35 or rsi_ant <= 35
        dejo_de_caer = rsi_act >= rsi_ant

        if dejo_de_caer and estuvo_sobrevendido:
            empresas_diario.append({
                'Ticker': ticker,
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1)})
            
    except Exception as e:
        print(f"Error en diario para {ticker}: {e}")
        continue

In [ ]:
# OUTPUT EN DIARIO.

print("\n" + "="*95)
print("PAPELES DETECTADOS EN DIARIO:")
if empresas_diario:
    df_final = pd.DataFrame(empresas_diario)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales diarias.")

In [ ]:
# INTERSECCION DE DATOS 60 MINUTOS Y 1 DÍA.

tickers_60 = [str(e['Ticker']).strip().upper() for e in empresas_60min]
ticker_1d = [str(e['Ticker']).strip().upper() for e in empresas_diario]

tickers_comunes = list(set(tickers_60) & set(ticker_1d))

print("Tickers comunes en ambas temporalidades:", tickers_comunes)


In [ ]:
# DESCARGA DE DATOS TEMPORALIDAD SEMANAL

tickers_7d = obtener_tickers("1wk" , "2y")

In [ ]:
# ANÁLISIS SEMANAL

empresas_semanal = []

for ticker in tickers_7d.columns.levels[0]:
    try:
        df = tickers_7d[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos.
        df.ta.rsi(length=10, append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables de validación
        rsi_act = df['RSI_10'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]

        # Lógica de filtrado.
        if rsi_act <= 44 and (rsi_act > rsi_ant):
        
            empresas_semanal.append({
                'Ticker': ticker,
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1),
                'MACD': round(macd_act, 3),
            })
    except:
        continue

print("\n" + "="*95)
print("ACTIVOS DETECTADOS EN SEMANAL.")
if empresas_semanal:
    df_final = pd.DataFrame(empresas_semanal)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales semanales.")

In [ ]:
#TICKERS REPETIDOS EN TRIPLE PANTALLA.

tickers_60 = [e['Ticker'] for e in empresas_60min]
tickers_1d = [e['Ticker'] for e in empresas_diario]
#tickers_7d = [e['Ticker'] for e in empresas_semanal]

tickers_comunes = list(set(tickers_60) & set(tickers_1d) 
                       #& set(tickers_7d)
                       )

print(tickers_comunes)